In [1]:
import pandas as pd 
import numpy as np 

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('../data/ABT_partition3.csv')

In [3]:
df

,class,active_region,start_time,end_time,TOTUSJH_Max,TOTUSJH_Min,TOTUSJH_Median,TOTUSJH_Mean,TOTUSJH_Std_dev,TOTUSJH_Variance,...,R_VALUE_Mean,R_VALUE_Std_dev,R_VALUE_Variance,R_VALUE_Skewness,R_VALUE_Kurtosis,R_VALUE_avg_abs_derivative_change,R_VALUE_last_value,R_VALUE_average_absolute_change,R_VALUE_quadratic_weighted_moving_average,R_VALUE_linear_weighted_moving_average
0,1,3311,NaN,NaN,2474.544114,1711.723109,1894.321713,2013.693434,229.075744,52475.696404,...,5.060391,0.033606,0.001129,-0.905727,-0.180802,0.001358,4.972729,0.010134,5.033443,5.044578
1,1,3311,NaN,NaN,2474.544114,1711.723109,1870.403576,1969.993291,209.692318,43970.868284,...,5.050524,0.041555,0.001727,-0.781983,-0.563198,0.001364,4.975827,0.010220,5.015907,5.029832
2,1,3311,NaN,NaN,2474.544114,1711.723109,1855.876895,1933.634818,186.147397,34650.853543,...,5.036840,0.054079,0.002925,-0.879172,-0.149674,0.001449,4.916409,0.011173,4.990972,5.009121
3,1,3311,NaN,NaN,2262.463905,1711.723109,1855.876895,1893.352970,129.896039,16872.981032,...,5.024660,0.058432,0.003414,-0.532526,-0.935352,0.001578,4.939300,0.011841,4.975388,4.993758
4,1,3311,NaN,NaN,2149.770882,1711.723109,1855.876895,1869.374332,91.243636,8325.401041,...,5.013803,0.057979,0.003362,-0.257532,-1.212933,0.001706,4.955240,0.013134,4.967622,4.983447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42505,0,4197,NaN,NaN,1451.188217,905.089295,1278.734747,1223.798091,184.251996,33948.797993,...,4.342240,0.102472,0.010501,-0.655453,-0.987051,0.003084,4.433733,0.022448,4.414694,4.394534
42506,0,4197,NaN,NaN,1496.725595,908.968506,1340.043645,1270.847993,170.001427,28900.485136,...,4.371060,0.094808,0.008989,-0.627758,-0.434919,0.003434,4.556813,0.024358,4.438512,4.419005
42507,0,4197,NaN,NaN,1531.719049,973.226004,1365.273177,1318.136835,148.428038,22030.882431,...,4.395604,0.085560,0.007320,-0.766081,0.357804,0.003455,4.520838,0.025715,4.455300,4.437921
42508,0,4197,NaN,NaN,1616.181064,1065.618839,1381.500941,1365.422127,133.373242,17788.421695,...,4.420139,0.070335,0.004947,-0.414842,-0.022657,0.003374,4.502935,0.024174,4.470856,4.455034


In [4]:
df  = df.drop(['start_time','end_time'],axis=1)

In [5]:
Y = df.copy()['class']
X = df.copy().drop(['class','active_region'],axis=1)

GROUP = df.copy()['active_region']

In [6]:
def add_gaussian_noise(X_train,y_train,minority=1,samples=30000,sigma=0.01, random_state=42):

    X_train = pd.DataFrame(X_train).copy()
    y_train = pd.Series(y_train).copy()


    minority_mask = (y_train == minority) # query T F for row in target minority 
    X_minority = X_train[minority_mask].copy()

    Sampled_X = X_minority.sample(n=samples,replace=True,random_state=random_state)

    #ensure numerica columns here 


    X_num = Sampled_X.select_dtypes(include=[np.number])
    feature_std = X_num.std(axis=0).replace(0,1e-16)

    np.random.seed(random_state)
    noise = np.random.normal(loc=0,scale=sigma,size=X_num.shape) * feature_std.values

    X_noise_num = X_num + noise

    X_noise = Sampled_X.copy()
    X_noise[X_num.columns] = X_noise_num

    y_noise = pd.Series([minority] * samples,index=X_noise.index)

    X_aug = pd.concat([X_train,X_noise], ignore_index=True)
    Y_aug = pd.concat([y_train,y_noise.reset_index(drop=True)],ignore_index=True)

    return X_aug,Y_aug

In [7]:
Y = df.copy()['class']
X = df.copy().drop(['class','active_region'],axis=1)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

from sklearn.metrics import roc_curve, auc
import plotly.express as px
import plotly.graph_objects as go

groups = GROUP

gkf = GroupKFold(n_splits=5)

oof_proba = pd.Series(index=Y.index, dtype=float)
fold_aucs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, Y, groups=groups), 1):
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = Y.iloc[tr_idx].copy(), Y.iloc[va_idx].copy()

    # noise ONLY on training fold
    X_tr, y_tr = add_gaussian_noise(X_tr, y_tr)

    # scale per fold (avoid leakage)
    scaler = StandardScaler()
    X_tr_scaled = pd.DataFrame(
        scaler.fit_transform(X_tr),
        columns=X_tr.columns,
        index=X_tr.index
    )
    X_va_scaled = pd.DataFrame(
        scaler.transform(X_va),
        columns=X_va.columns,
        index=X_va.index
    )

    # class imbalance weight from training fold
    neg = (y_tr == 0).sum()
    pos = (y_tr == 1).sum()
    spw = neg / max(pos, 1)

    model = XGBClassifier(
        n_estimators=5000,
        learning_rate=0.01,
        max_depth=6,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=spw,
        seed =42,
        random_state=42,
        n_jobs=3,
    )

    model.fit(
        X_tr_scaled, y_tr,
        eval_set=[(X_va_scaled, y_va)],
        verbose=False
    )

    proba_va = model.predict_proba(X_va_scaled)[:, 1]
    oof_proba.loc[X_va_scaled.index] = proba_va

    auc = roc_auc_score(y_va, proba_va)
    fold_aucs.append(auc)

    print(f"Fold {fold} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"OOF AUC:  {roc_auc_score(Y.loc[oof_proba.index], oof_proba):.4f}")

# threshold metrics
threshold = 0.5
oof_pred = (oof_proba >= threshold).astype(int)

print(confusion_matrix(Y.loc[oof_proba.index], oof_pred))
print(classification_report(Y.loc[oof_proba.index], oof_pred))



Fold 1 AUC: 0.9214
Fold 2 AUC: 0.9081
Fold 3 AUC: 0.9348
Fold 4 AUC: 0.9431
Fold 5 AUC: 0.9308

Mean AUC: 0.9276 ± 0.0120
OOF AUC:  0.9278
[[32545  2217]
 [ 2475  5273]]
              precision    recall  f1-score   support

           0       0.93      0.94      0.93     34762
           1       0.70      0.68      0.69      7748

    accuracy                           0.89     42510
   macro avg       0.82      0.81      0.81     42510
weighted avg       0.89      0.89      0.89     42510



TypeError: 'float' object is not callable

In [11]:
# Plotly Confusion Matrix
cm = confusion_matrix(Y.loc[oof_proba.index], oof_pred)

cm_fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale="Blues",
    x=["Predicted 0", "Predicted 1"],
    y=["Actual 0", "Actual 1"],
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="Confusion Matrix"
)

cm_fig.update_layout(
    xaxis=dict(side="bottom"),
    yaxis=dict(autorange="reversed")
)


# save html
cm_fig.write_html("../webapp/graphs/xgboost_confusion_matrix.html")


# Plotly ROC Curve
fpr, tpr, _ = roc_curve(Y.loc[oof_proba.index], oof_proba)
roc_auc = roc_auc_score(Y.loc[oof_proba.index], oof_proba)

roc_fig = go.Figure()

roc_fig.add_trace(go.Scatter(
    x=fpr,
    y=tpr,
    mode="lines",
    name=f"ROC curve (AUC = {roc_auc:.4f})"
))

roc_fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Random classifier",
    line=dict(dash="dash")
))

roc_fig.update_layout(
    title="ROC Curve",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white"
)


# save html
roc_fig.write_html("../webapp/graphs/xgboost_roc_curve.html")


In [12]:
importances = model.feature_importances_
feature_names = X_tr_scaled.columns

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(importance_df)

                               feature  importance
300                        R_VALUE_Min    0.256805
8    TOTUSJH_avg_abs_derivative_change    0.039186
92                           TOTFZ_Min    0.032920
101      TOTFZ_average_absolute_change    0.032880
9                   TOTUSJH_last_value    0.025507
..                                 ...         ...
98                      TOTFZ_Kurtosis    0.000259
123                      EPSZ_Skewness    0.000241
111                   MEANPOT_Kurtosis    0.000234
163                   MEANGAM_Kurtosis    0.000223
46                    TOTUSJZ_Kurtosis    0.000188

[312 rows x 2 columns]


In [ ]:
importance_df.to_csv('../webapp/XGBoost/XGFeatureImportance.csv')

In [17]:

top_22_features = pd.DataFrame({
    'feature': X_tr_scaled.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(22)['feature'].tolist()

top_22_features

['R_VALUE_Min',
 'TOTUSJH_avg_abs_derivative_change',
 'TOTFZ_Min',
 'TOTFZ_average_absolute_change',
 'TOTUSJH_last_value',
 'TOTPOT_Variance',
 'SAVNCPP_Variance',
 'TOTUSJH_Max',
 'ABSNJZH_Min',
 'SAVNCPP_average_absolute_change',
 'ABSNJZH_average_absolute_change',
 'TOTUSJH_Variance',
 'TOTFZ_Variance',
 'TOTFZ_last_value',
 'R_VALUE_Variance',
 'TOTPOT_Std_dev',
 'MEANPOT_Mean',
 'ABSNJZH_Variance',
 'TOTBSQ_Variance',
 'MEANPOT_Variance',
 'MEANGBZ_Max',
 'TOTUSJH_Std_dev']

'R_VALUE_Min'

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

Y = df.copy()['class']
X = df.copy().drop(['class','active_region'],axis=1)

X = X[top_22_features]


GROUP = df.copy()['active_region']



X_train,X_test,y_train,y_test = train_test_split(X,Y,random_state=42,test_size=0.2,shuffle=True,stratify=Y)

scaler = StandardScaler()


# Use GroupShuffleSplit to split by active region
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, Y, groups=GROUP))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]

# check for  no overlap between ARS
train_regions = set(GROUP.iloc[train_idx])
test_regions = set(GROUP.iloc[test_idx])
print(f"Train regions: {len(train_regions)}")
print(f"Test regions: {len(test_regions)}")
print(f"Overlap: {len(train_regions & test_regions)}")  # Should be 0! if not then leakage 

#scaling is needed for XGBoost in this scenario. 
#theres datapoints that are too massive and throws an error so we need to scale and then add gausasin noise
scaler = StandardScaler()
X_train, y_train = add_gaussian_noise(X_train, y_train,samples=5000)
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# class imbalance weight from training fold
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / max(pos, 1)

model = XGBClassifier(
    n_estimators=5000,
    learning_rate=0.01,
    max_depth=6,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=spw,
    seed=42,
    random_state=42,
    n_jobs=3,
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)


from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC AUC score
print(f"\nROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")





# Evaluate on held-out test set
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"\nROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

# model deplotmnet
import pickle

# Save the scaler and model
with open('../models/XGBoost/scaler_v2.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
model.save_model('../models/XGBoost/XGBoostModel_v2.ubj')

# Also save feature names for validation
with open('../models/XGBoost/feature_names_v2.pkl', 'wb') as f:
    pickle.dump(X_train.columns.tolist(), f)

print("\n Model and scaler saved!")